### Phase 4: Feature Engineering and Dimensionality Reduction

Here we apply K-Means and PCA, and crucially extract a spatial Group array (cenlon_cenlat) to prevent Spatial Leakage downstream.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import joblib
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_parquet("../final_datasets/clean_datasets/1_data_preparation_dataset.parquet", engine="fastparquet")


In [3]:

# extracting Groups using Longitude and Latitude to uniquely identify glaciers
groups = df["cenlon"].astype(str) + "_" + df["cenlat"].astype(str)
np.save("../final_datasets/clean_datasets/groups.npy", groups.values)


In [4]:

# dropping identifiers if any exist
to_drop = ["glacier_name", "rgi_id", "country_left", "country_right", "region", "dem_source"]


In [5]:

X = df.drop(columns=to_drop + ["actual_mass_change"], errors="ignore")
y = (df["actual_mass_change"] > 0).astype(int).values

# dropping collinearity features
collinear_features = ["stl1", "stl2", "stl3", "stl4", "strd"]
X = X.drop(columns=collinear_features, errors="ignore")


In [6]:
joblib.dump(collinear_features, "../pkl/collinear_dropped.pkl")


['../pkl/collinear_dropped.pkl']

In [7]:

X_num = X.select_dtypes(include=[np.number]).fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)

In [8]:
joblib.dump(scaler, "../pkl/scaler.pkl")


['../pkl/scaler.pkl']

In [9]:
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

In [10]:

joblib.dump(kmeans, "../pkl/kmeans_model.pkl")
X_scaled_with_cluster = np.column_stack((X_scaled, clusters))

In [11]:
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled_with_cluster)



In [12]:
joblib.dump(pca, "../pkl/pca_model.pkl")


['../pkl/pca_model.pkl']

In [13]:

np.save("../final_datasets/clean_datasets/X_pca.npy", X_pca)
np.save("../final_datasets/clean_datasets/y.npy", y)
print(f"PCA Shape: {X_pca.shape}")

PCA Shape: (12112, 18)


In [15]:
X_pca

array([[ 9.01427824, -0.15179721,  0.95700203, ..., -0.11980095,
         0.62596972,  0.26049275],
       [ 8.32288624,  0.72055263,  0.60708913, ..., -0.15928319,
         0.83096121,  0.14760972],
       [10.26491022,  0.38688669,  1.2946023 , ..., -0.5058699 ,
         0.7924034 , -0.05096687],
       ...,
       [-5.08168899,  0.8964334 , -2.57994415, ...,  1.00216273,
         0.94479361, -0.95775079],
       [-5.34715808,  1.34448506, -1.29711286, ..., -0.23695882,
        -0.27037617, -0.05534036],
       [ 5.51093203, -0.21543244, -2.47540656, ...,  0.31439815,
        -0.30790404,  0.09211254]], shape=(12112, 18))

In [16]:
y

array([0, 0, 0, ..., 0, 0, 0], shape=(12112,))